In [ ]:
from pathlib import Path

import pandas as pd
import uproot

from utils import TREE_NAME, branch_summary, sample_label

pd.set_option("display.max_columns", 120)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
INPUT_DIR = PROJECT_ROOT / "output" / "florian"
INPUT_FILES = [
    INPUT_DIR / "ZKK.root",
    INPUT_DIR / "Zmumu.root",
    INPUT_DIR / "Zpipi.root",
]
PLOTS_DIR = PROJECT_ROOT / "plots" / "florian"


def _contains_label(labels: str, label: str) -> bool:
    return label in labels.split(", ") if labels else False


def sanity_summary(paths: list[Path], branch_table: pd.DataFrame) -> pd.DataFrame:
    branches = branch_table["branch"].astype(str)
    rows = []

    for path in paths:
        tree = uproot.open(path)[TREE_NAME]
        file_name = sample_label(path)
        missing = branch_table[
            branch_table["missing_files"].apply(lambda labels: _contains_label(labels, file_name))
        ]
        empty = branch_table[
            branch_table["empty_files"].apply(lambda labels: _contains_label(labels, file_name))
        ]

        rows.append(
            {
                "file": file_name,
                "sample": file_name,
                "entries": tree.num_entries,
                "branches_in_file": len(tree.keys()),
                "summary_branches": len(branch_table),
                "sdst_branches": int(branches.str.startswith("SDST_").sum()),
                "raw_sdst_branches": int(branches.str.startswith("RAWSDST_").sum()),
                "raw_fadana_branches": int(branches.str.startswith("RAWFADANA_").sum()),
                "present_branches": len(branch_table) - len(missing),
                "missing_branch_count": len(missing),
                "missing_branches": ", ".join(missing["branch"]),
                "empty_branch_count": len(empty),
                "empty_branches": ", ".join(empty["branch"]),
                "numeric_branches": int(branch_table["is_numeric"].sum()),
                "nan_values": int(branch_table["nan_count"].sum()),
            }
        )

    return pd.DataFrame(rows)


files = [Path(path) for path in INPUT_FILES]
missing_files = [path for path in files if not path.exists()]
assert not missing_files, "Missing input ROOT files:\n" + "\n".join(str(path) for path in missing_files)

PLOTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Found {len(files)} ROOT files")
print(f"Writing plots and tables to {PLOTS_DIR}")


In [2]:
branch_table = branch_summary(files)
summary = sanity_summary(files, branch_table)

summary.to_csv(PLOTS_DIR / "sanity_summary.csv", index=False)
display(summary)


,file,sample,job,entries,branches_in_file,summary_branches,sdst_branches,raw_sdst_branches,raw_fadana_branches,present_branches,missing_branch_count,missing_branches,empty_branch_count,empty_branches,numeric_branches,nan_values
0,ZKK,ZKK,ZKK,1800,538,538,344,115,77,538,0,,0,,492,13770
1,Zmumu,Zmumu,Zmumu,1800,538,538,344,115,77,538,0,,0,,492,13770
2,Zpipi,Zpipi,Zpipi,1800,538,538,344,115,77,538,0,,0,,492,13770
